# Trustworthy AI Resume Screener v2 — Mac MPS / Colab CUDA
This notebook runs the same experiment package on Apple Silicon MPS, NVIDIA CUDA, or CPU, and writes resumable JSONL checkpoints. Report metrics must come from the `qwen` backend.

In Colab, upload this notebook and run the first code cell. If the project is not already under `/content`, it will ask for a project `.zip`, extract it, and locate `run_experiment.py` automatically. No GitHub clone is required.

In [ ]:
from pathlib import Path
import shutil, zipfile

def find_project(search_root):
    search_root = Path(search_root)
    if (search_root / 'run_experiment.py').exists():
        return search_root
    if not search_root.exists():
        return None
    matches = [p for p in search_root.rglob('run_experiment.py') if '__MACOSX' not in p.parts]
    return matches[0].parent if matches else None

# Local: use the current repository. Colab: first look for an already uploaded/extracted project.
PROJECT_ROOT = find_project(Path.cwd())
if PROJECT_ROOT is None and Path('/content').exists():
    PROJECT_ROOT = find_project('/content')

# If Colab still has no project, prompt for one project ZIP and extract it automatically.
if PROJECT_ROOT is None:
    try:
        from google.colab import files
        print('Upload the ZIP containing run_experiment.py and the src/ folder.')
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
        if not zip_names:
            raise FileNotFoundError('No .zip file was uploaded.')
        extract_root = Path('/content/resume_project_upload')
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True)
        with zipfile.ZipFile(Path.cwd() / zip_names[0]) as archive:
            archive.extractall(extract_root)
        PROJECT_ROOT = find_project(extract_root)
    except ImportError:
        pass

assert PROJECT_ROOT is not None, (
    'Project not found. On Mac, open the notebook from the repository root. '
    'In Colab, upload a ZIP containing run_experiment.py, requirements.txt, and src/.'
)
print('Project root:', PROJECT_ROOT)
REQUIREMENTS = str(PROJECT_ROOT / 'requirements.txt')
%pip install -q -r $REQUIREMENTS

In [ ]:
import sys, torch
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if torch.cuda.is_available():
    DEVICE = 'cuda'
    DEVICE_NAME = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    DEVICE_NAME = 'Apple Silicon MPS'
else:
    DEVICE = 'cpu'
    DEVICE_NAME = 'CPU (slow; CUDA or MPS recommended)'
print('Selected device:', DEVICE, '-', DEVICE_NAME)

In [ ]:
from trustworthy_resume import ExperimentConfig, run_experiment
config = ExperimentConfig(
    backend='qwen', device=DEVICE, model_name='Qwen/Qwen3-0.6B',
    output_root=str(PROJECT_ROOT / 'outputs'), run_id=f'qwen_v2_n100_{DEVICE}',
    num_candidates=100, fairness_templates_per_attribute=10,
    repeatability_samples=10, repeatability_repeats=3, use_cache=True,
)
metrics = run_experiment(config)
metrics

In [ ]:
import pandas as pd, json
output_dir = config.output_dir
display(pd.read_csv(output_dir / 'robustness_summary.csv'))
display(pd.read_csv(output_dir / 'fairness_bootstrap_ci.csv'))
display(pd.read_csv(output_dir / 'explainability_audit.csv').describe(include='all'))
display(pd.read_csv(output_dir / 'repeatability_results.csv'))
display(pd.read_csv(output_dir / 'governance_summary.csv'))
print(json.loads((output_dir / 'headline_metrics.json').read_text()))